<div style="background: linear-gradient(135deg, #1a2a6c, #2c3e91); padding: 30px; border-radius: 12px; color: white; font-family: 'Segoe UI', Arial, sans-serif;">
  <h1 style="margin: 0; font-size: 28px;">📡 Automatic Modulation Classification Engine</h1>
  <h3 style="margin: 8px 0 0 0; font-weight: 400; color: #dcdfff;">Using 1D Convolutional Neural Networks</h3>
  <p style="margin-top: 15px; font-size: 14px; color: #c9cdf7;">
    <b>Author:</b> R. M. S. H. Ratnayake &nbsp;|&nbsp; <b>Date:</b> August 2026
  </p>
</div>

<div style="background: #f4f6fb; padding: 20px; border-radius: 10px; margin-top: 15px; border-left: 5px solid #2c3e91;">
  <h3 style="color: #1a2a6c;">🎯 Overview</h3>
  <p style="font-size: 15px; line-height: 1.6;">
    This notebook implements an end-to-end deep learning pipeline for <b>Automatic Modulation Classification (AMC)</b>,
    classifying <b>11 analog and digital modulation schemes</b> directly from raw complex I/Q time-series samples.
    Unlike traditional likelihood-based or 2D image-based (spectrogram) approaches, this system uses a lightweight
    <b>1D CNN</b> to process <code>2 × N</code> I/Q tensors directly, reducing computational overhead while preserving
    phase and amplitude information critical for classification.
  </p>
</div>

<div style="background: #f4f6fb; padding: 20px; border-radius: 10px; margin-top: 15px; border-left: 5px solid #2c3e91;">
  <h3 style="color: #1a2a6c;">📋 Objectives</h3>
  <ul style="font-size: 15px; line-height: 1.8;">
    <li>Build a full pipeline to classify 11 modulation schemes using the <b>RadioML 2016.10A</b> benchmark dataset</li>
    <li>Implement a lightweight 1D CNN for raw complex I/Q inputs</li>
    <li>Evaluate accuracy across SNR levels from <b>−20 dB to +18 dB</b></li>
    <li>Apply <b>INT8 quantization</b> for low-latency, edge-deployable inference</li>
  </ul>
</div>

<div style="background: #f4f6fb; padding: 20px; border-radius: 10px; margin-top: 15px; border-left: 5px solid #2c3e91;">
  <h3 style="color: #1a2a6c;">🏗️ Model Architecture</h3>
  <table style="width: 100%; border-collapse: collapse; font-size: 14px; margin-top: 10px;">
    <tr style="background: #1a2a6c; color: white;">
      <th style="padding: 8px; text-align: left;">Layer</th>
      <th style="padding: 8px;">Kernel</th>
      <th style="padding: 8px;">Stride</th>
      <th style="padding: 8px;">Output Shape</th>
    </tr>
    <tr style="background: #e8eaf5;"><td style="padding: 8px;">Input Tensor</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">(2, 128)</td></tr>
    <tr><td style="padding: 8px;">Conv1D + BN + ReLU</td><td style="padding: 8px; text-align:center;">7</td><td style="padding: 8px; text-align:center;">1</td><td style="padding: 8px; text-align:center;">(64, 128)</td></tr>
    <tr style="background: #e8eaf5;"><td style="padding: 8px;">MaxPool1D</td><td style="padding: 8px; text-align:center;">2</td><td style="padding: 8px; text-align:center;">2</td><td style="padding: 8px; text-align:center;">(64, 64)</td></tr>
    <tr><td style="padding: 8px;">Conv1D + BN + ReLU</td><td style="padding: 8px; text-align:center;">5</td><td style="padding: 8px; text-align:center;">1</td><td style="padding: 8px; text-align:center;">(128, 64)</td></tr>
    <tr style="background: #e8eaf5;"><td style="padding: 8px;">MaxPool1D</td><td style="padding: 8px; text-align:center;">2</td><td style="padding: 8px; text-align:center;">2</td><td style="padding: 8px; text-align:center;">(128, 32)</td></tr>
    <tr><td style="padding: 8px;">Conv1D + BN + ReLU</td><td style="padding: 8px; text-align:center;">3</td><td style="padding: 8px; text-align:center;">1</td><td style="padding: 8px; text-align:center;">(256, 32)</td></tr>
    <tr style="background: #e8eaf5;"><td style="padding: 8px;">AdaptiveAvgPool1D</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">(256, 1)</td></tr>
    <tr><td style="padding: 8px;">Dense + Dropout(0.5)</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">(128)</td></tr>
    <tr style="background: #e8eaf5;"><td style="padding: 8px;">Dense Output</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">–</td><td style="padding: 8px; text-align:center;">(11)</td></tr>
  </table>
</div>

<div style="background: #eef7ee; padding: 20px; border-radius: 10px; margin-top: 15px; border-left: 5px solid #2e7d32;">
  <h3 style="color: #1a2a6c;">✅ Expected Deliverables</h3>
  <ol style="font-size: 15px; line-height: 1.8;">
    <li>Modular PyTorch pipeline (data loading, training, evaluation)</li>
    <li>Accuracy-vs-SNR performance curves (target &gt;90% at SNR ≥ 0 dB)</li>
    <li>Quantized ONNX model for edge deployment</li>
    <li>Documented experimental results</li>
  </ol>
</div>

In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
